# 05 - Export results for the website

Turns the outputs of notebooks 03 and 04 into one file, `site_data.json`, that the website reads. Nothing is trained or tuned here, and no new test-set number is produced: the test predictions come from notebook 04.

It exports:
- every test match, over by over: score, wickets, the model's probability and baseline B's probability (for the match explorer);
- the results table, calibration curve, log loss by over and feature importance (for the results charts);
- the coefficients of a logistic model so the browser can compute the "what if" panel without a server.

**Run notebook 04 first.** This notebook stops if `test_results.json` does not exist.

## 1. Settings (edit the paths)

In [2]:
TRAIN_PATH     = r"../data/split/train.csv"
VAL_PATH       = r"../data/split/validation.csv"
TEST_PATH      = r"../data/split/test.csv"
SELECTION_PATH = r"../results/model_selection.json"
RESULTS_DIR    = r"../results"            # notebook 03 outputs (validation_table.csv)
FINAL_DIR      = r"../results/final"      # notebook 04 outputs
SITE_DATA_PATH = r"../site/data/site_data.json"
REPO_URL       = "https://github.com/tashakaggarwal/es335-2026"     # e.g. "https://github.com/you/es335-ipl" (shown in the site footer)
AUTHOR         = ""                    # your name (shown on the site)
RANDOM_STATE   = 0

## 2. Load

In [3]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

final_dir = Path(FINAL_DIR)
assert (final_dir / "test_results.json").exists(), "Run notebook 04 first (test_results.json not found)."
sel = json.loads(Path(SELECTION_PATH).read_text())
res = json.loads((final_dir / "test_results.json").read_text())
assert res["model"] == sel["model"] and res["features"] == sel["features"], "model_selection.json and test_results.json disagree"

rd = lambda p: pd.read_csv(p, dtype={"match_id": str})
train, val, test = rd(TRAIN_PATH), rd(VAL_PATH), rd(TEST_PATH)
pred = rd(final_dir / "test_predictions.csv")
table = pd.read_csv(final_dir / "test_table.csv", index_col=0)
imp = pd.read_csv(final_dir / "test_permutation_importance.csv", index_col=0).iloc[:, 0]
assert (pred.match_id.to_numpy() == test.match_id.to_numpy()).all() and (pred.over.to_numpy() == test.over.to_numpy()).all(), \
    "test_predictions.csv is not in the same row order as the test file"

col = lambda prefix: next(c for c in pred.columns if c.startswith(prefix))
P_MODEL, P_RULE, P_BYOVER = col("FINAL"), col("B "), col("A2")
y = pred.chasing_team_won.to_numpy()
print(f"model: {sel['model']} on '{sel['feature_set']}' | test: {test.match_id.nunique()} matches, {len(test)} rows")

model: logreg on 'D +batters+impact' | test: 138 matches, 2434 rows


## 3. Matches for the explorer

In [4]:
def iso(s):
    s = s.astype(str)
    if s.str.fullmatch(r"\d{4}-\d{2}-\d{2}").all():
        return pd.to_datetime(s, format="%Y-%m-%d")
    if s.str.fullmatch(r"\d{2}-\d{2}-\d{2}").all():
        return pd.to_datetime(s, format="%d-%m-%y")
    return pd.to_datetime(s)

T = test.copy()
T["p"], T["pB"] = pred[P_MODEL].to_numpy(), pred[P_RULE].to_numpy()
venue_col = "venue_clean" if "venue_clean" in T.columns else ("venue" if "venue" in T.columns else None)
T["date_iso"] = iso(T.date).dt.strftime("%Y-%m-%d") if "date" in T.columns else ""

matches = []
for mid, g in T.sort_values(["match_id", "over"]).groupby("match_id", sort=False):
    f = g.iloc[0]
    matches.append({
        "id": mid, "season": int(f.season), "date": f.date_iso,
        "chaser": f.batting_team if "batting_team" in g else "Chasing team",
        "bowler": f.bowling_team if "bowling_team" in g else "Defending team",
        "venue": f[venue_col] if venue_col else "",
        "target": int(f.target), "won": int(f.chasing_team_won),
        "overs": [[int(r.over), int(r.runs_so_far), int(r.wickets_lost), round(float(r.p), 4), round(float(r.pB), 4)]
                  for r in g.itertuples()],
    })
matches.sort(key=lambda m: (m["date"], m["id"]))
print(len(matches), "matches; first:", matches[0]["chaser"], "v", matches[0]["bowler"], matches[0]["date"])

138 matches; first: Royal Challengers Bengaluru v Kolkata Knight Riders 2025-03-22


## 4. Results, calibration, by-over loss, importance

In [5]:
tbl = []
for name, r in table.iterrows():
    tbl.append({"name": name.strip(), "log_loss": float(r.log_loss), "brier": float(r.brier), "accuracy": float(r.accuracy),
                "ci": r["95% CI"], "diff": None if pd.isna(r["diff vs B"]) else float(r["diff vs B"]),
                "diff_ci": r["diff 95% CI"] if isinstance(r["diff 95% CI"], str) else ""})

frac, mean_p = calibration_curve(y, np.clip(T.p, 1e-6, 1 - 1e-6), n_bins=8, strategy="quantile")
cal_model = [[round(float(a), 4), round(float(b), 4)] for a, b in zip(mean_p, frac)]
cal_rule = [[round(float(p), 4), round(float(g.chasing_team_won.mean()), 4), int(len(g))] for p, g in T.groupby("pB")]

def by_over_loss(p):
    p = np.clip(p, 1e-6, 1 - 1e-6); l = -(y * np.log(p) + (1 - y) * np.log(1 - p))
    return pd.Series(l).groupby(T.over.to_numpy()).mean()
bo = pd.DataFrame({"model": by_over_loss(T.p.to_numpy()), "rule": by_over_loss(T.pB.to_numpy()),
                   "by_over": by_over_loss(pred[P_BYOVER].to_numpy())})
by_over = [{"over": int(i), **{k: round(float(v), 4) for k, v in r.items()}} for i, r in bo.iterrows()]

importance = [{"feature": k, "value": round(float(v), 5)} for k, v in imp.sort_values(ascending=False).items()]

val_tbl = None
vp = Path(RESULTS_DIR) / "validation_table.csv"
if vp.exists():
    v = pd.read_csv(vp, index_col=0)
    val_tbl = {k.strip(): float(r.log_loss) for k, r in v.iterrows()}

## 5. The "what if" model

The browser needs a model it can evaluate by hand. A logistic regression is just a weighted sum, so its standardisation constants and coefficients are exported. If your selected model is logistic regression, that exact model is used. If it is gradient boosting, a logistic regression on the ten core and derived features is fitted instead and the site says so.

Features the visitor cannot set in the panel (momentum, batters, and so on) are held at their training average, which contributes zero after standardising.

In [6]:
BASE10 = ["runs_so_far", "target", "wickets_lost", "balls_left", "current_run_rate", "required_run_rate",
          "run_rate_diff", "runs_needed_frac", "resources", "runs_needed_per_wicket"]
full = pd.concat([train, val], ignore_index=True)
yf = full.chasing_team_won.to_numpy()

if sel["model"] == "logreg":
    feats, C_, same_as_final = sel["features"], sel["params"]["C"], True
else:
    feats, C_, same_as_final = BASE10, 0.1, False

pre = ColumnTransformer([("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                                           ("scale", StandardScaler())]), feats)])
lr = Pipeline([("pre", pre), ("clf", LogisticRegression(C=C_, max_iter=5000))]).fit(full[feats], yf)
sc = lr.named_steps["pre"].named_transformers_["num"].named_steps["scale"]
clf = lr.named_steps["clf"]

whatif = {
    "features": feats, "mean": [float(x) for x in sc.mean_], "scale": [float(x) for x in sc.scale_],
    "coef": [float(x) for x in clf.coef_[0]], "intercept": float(clf.intercept_[0]),
    "settable": [f for f in feats if f in BASE10],
    "held_at_average": [f for f in feats if f not in BASE10],
    "same_as_final_model": same_as_final,
    "target_range": [int(full.target.quantile(0.01)), int(max(full.target.max(), test.target.max()))],
}


rule_f = (full.run_rate_diff <= 0)
p_win_f, p_lose_f = round(float(yf[rule_f].mean()), 4), round(float(yf[~rule_f].mean()), 4)
overs_df = pd.read_csv(r"../data/parsed/overs.csv", dtype={"match_id": str})
s1 = overs_df[["date", "season", "striker", "striker_past_sr", "striker_past_balls", "batting_team"]].rename(columns={"striker": "name", "striker_past_sr": "sr", "striker_past_balls": "balls", "batting_team": "team"})
s2 = overs_df[["date", "season", "non_striker", "non_striker_past_sr", "non_striker_past_balls", "batting_team"]].rename(columns={"non_striker": "name", "non_striker_past_sr": "sr", "non_striker_past_balls": "balls", "batting_team": "team"})
s_all = pd.concat([s1, s2]).dropna(subset=["name"]).sort_values(["date", "season"])
lat = s_all.groupby("name").last().reset_index()
lat["shrunk_sr"] = ((lat["sr"].fillna(125.0) * lat["balls"].fillna(0) + 125.0 * 60.0) / (lat["balls"].fillna(0) + 60.0)).round(2)
batters_map = dict(zip(lat["name"], [float(x) for x in lat["shrunk_sr"]]))
rosters_map = {t: g.groupby("name").size().sort_values(ascending=False).head(12).index.tolist() for t, g in s_all[s_all["season"] >= 2025].groupby("team")}
whatif["batters"] = batters_map
whatif["rosters"] = rosters_map
whatif["rule_probs"] = [p_lose_f, p_win_f]

groups_all = json.loads(Path(r"../data/features/feature_groups.json").read_text())
all27 = [f for g in groups_all.values() for f in g]
pre27 = ColumnTransformer([("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), all27)])
lr27 = Pipeline([("pre", pre27), ("clf", LogisticRegression(C=0.1, max_iter=5000))]).fit(full[all27], yf)
sc27 = lr27.named_steps["pre"].named_transformers_["num"].named_steps["scale"]
clf27 = lr27.named_steps["clf"]
df_all = pd.read_csv(r"../data/features/features_all.csv", dtype={"match_id": str})
mt = df_all.groupby("match_id").first().reset_index().sort_values(["date", "match_id"]).reset_index(drop=True)
p0_all = float(mt["chasing_team_won"].mean())
venues_map = {v: round(float((g["chasing_team_won"].sum() + 10.0 * p0_all) / (len(g) + 10.0)), 4) for v, g in mt.groupby("venue_clean")}
hg = pd.concat([mt[mt["chasing_team_home"] == 1].assign(team=lambda x: x["batting_team"], hw=lambda x: x["chasing_team_won"]), mt[(mt["bowling_team_home"] == 1) & (mt["chasing_team_home"] != 1)].assign(team=lambda x: x["bowling_team"], hw=lambda x: 1 - x["chasing_team_won"])])
ph_all = float(hg["hw"].mean())
team_home_edge = {t: round(float((g["hw"].sum() + 10.0 * ph_all) / (len(g) + 10.0) - ph_all), 4) for t, g in hg.groupby("team")}
team_home_venues = {t: g["venue_clean"].value_counts().index.tolist() for t, g in hg.groupby("team")}
team_recent_wins = {t: int(np.where(mt[(mt["batting_team"] == t) | (mt["bowling_team"] == t)].tail(5)["batting_team"] == t, mt[(mt["batting_team"] == t) | (mt["bowling_team"] == t)].tail(5)["chasing_team_won"], 1 - mt[(mt["batting_team"] == t) | (mt["bowling_team"] == t)].tail(5)["chasing_team_won"]).sum()) for t in sorted(set(mt["batting_team"]) | set(mt["bowling_team"]))}
whatif["live_model"] = {"features": all27, "mean": [float(x) for x in sc27.mean_], "scale": [float(x) for x in sc27.scale_], "coef": [float(x) for x in clf27.coef_[0]], "intercept": float(clf27.intercept_[0])}
whatif["venues"] = venues_map
whatif["team_home_edge"] = team_home_edge
whatif["team_home_venues"] = team_home_venues
whatif["team_recent_wins"] = team_recent_wins
whatif["default_season_avg_target"] = round(float(mt[mt["season"] == mt["season"].max()]["target"].mean()), 1)

# check: the hand-computed logit reproduces scikit-learn on real rows
Z = (test[feats].fillna(full[feats].median()).to_numpy() - np.array(whatif["mean"])) / np.array(whatif["scale"])
mine = 1 / (1 + np.exp(-(Z @ np.array(whatif["coef"]) + whatif["intercept"])))
theirs = lr.predict_proba(test[feats])[:, 1]
print("max |hand-computed - sklearn| =", float(np.abs(mine - theirs).max()))
assert np.abs(mine - theirs).max() < 1e-6
print("what-if model:", "the final model" if same_as_final else "simplified logistic regression on 10 features")
print("held at average:", whatif["held_at_average"])

max |hand-computed - sklearn| = 0.0
what-if model: the final model
held at average: ['runs_last_3', 'wkts_last_3', 'boundaries_last_3', 'acceleration', 'balls_since_last_wicket', 'bat_sr_mean', 'bat_balls_sum', 'bat_balls_max', 'chasing_impact_used', 'bowling_impact_used']


## 6. Write the file

In [7]:
fm = full.groupby("match_id").target.first(); tm = test.groupby("match_id").target.first()
site = {
    "meta": {
        "author": AUTHOR, "repo": REPO_URL,
        "model": sel["model"], "feature_set": sel["feature_set"], "n_features": len(sel["features"]),
        "fitted_seasons": res["fitted_on_seasons"], "test_seasons": res["test_seasons"],
        "test_matches": res["test_matches"], "test_rows": res["test_rows"],
        "train_matches": int(full.match_id.nunique()),
        "mean_target_fit": float(fm.mean()), "mean_target_test": float(tm.mean()),
        "calibration_slope": res["calibration_slope"], "calibration_intercept": res["calibration_intercept"],
    },
    "results": tbl, "validation_log_loss": val_tbl,
    "calibration": {"model": cal_model, "rule": cal_rule},
    "by_over": by_over, "importance": importance,
    "whatif": whatif, "matches": matches,
}
out = Path(SITE_DATA_PATH); out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(site, separators=(",", ":")))
print(f"wrote {out.resolve()}  ({out.stat().st_size/1024:.0f} KB, {len(matches)} matches)")

wrote C:\Users\LEGION\OneDrive\Documents\GitHub\es335-2026\site\data\site_data.json  (86 KB, 138 matches)


## 7. Next
Put `site_data.json` in `site/data/` next to `index.html`, then publish the `site` folder with GitHub Pages (Settings > Pages > deploy from a branch > choose the folder). Open the page and check: the explorer loads, the numbers in the results table match your notebook 04 output, and the footer links work.